# チュートリアル 11: Agent Skills — モジュラーな知識拡張

## 学習目標
このノートブックを完了すると、以下ができるようになります:
- Agent Skills の概念と **Progressive Disclosure パターン** を理解する
- `SKILL.md` ファイルによるスキル定義方法を習得する
- `SkillsProvider` でエージェントにスキルを付与する
- スキルの **Advertise → Load → Read Resources** の 3 段階動作を確認する
- 複数スキルを持つエージェントを構築する

## 主要概念

### Agent Skills とは？

**Agent Skills** は [Agent Skills 仕様](https://agentskills.io/) に基づくモジュラーな知識パッケージです。  
エージェントの instructions に全てを詰め込む代わりに、**必要な時に必要な知識だけ** をロードさせます。

### なぜ Skills が必要か？

| 従来のアプローチ | Agent Skills |
|---|---|
| 全ドメイン知識を instructions に記載 | 名前+説明だけ広告（~100トークン/スキル） |
| コンテキストウィンドウを圧迫 | オンデマンドで詳細をロード |
| スケールしない（知識が増えると破綻） | スキルを追加するだけでスケール |
| 知識の更新 = コード修正 | `SKILL.md` ファイルを更新するだけ |

### Progressive Disclosure パターン

```
1. Advertise（広告）
   └─ スキル名 + 説明をシステムプロンプトに注入（軽量）

2. Load（ロード）
   └─ エージェントが load_skill ツールを呼び出し → SKILL.md 本文を取得

3. Read Resources（リソース読込）
   └─ エージェントが read_skill_resource ツールで FAQ/テンプレート等を取得
```

### スキルのディレクトリ構造

```
skills/
├── travel-planner/
│   ├── SKILL.md              ← メインの指示書 (YAML frontmatter + 本文)
│   ├── references/
│   │   └── TRAVEL_FAQ.md     ← 参照ドキュメント
│   └── assets/
│       └── itinerary-template.md  ← テンプレート
└── code-reviewer/
    ├── SKILL.md
    └── references/
        └── CHECKLIST.md
```

### SKILL.md の形式

```markdown
---
name: travel-planner
description: 旅行の計画と予約に関するガイダンスを提供します。
---

# 本文（エージェント向け詳細指示）
ここに詳細なドメイン知識を記述...
参照: [references/FAQ.md](references/FAQ.md)
テンプレート: [assets/template.md](assets/template.md)
```

---

**このチュートリアルの前提**: `agent-framework >= 1.0.0rc2` (Agent Skills は rc2 で新規追加)

## ステップ 1: セットアップとインポート

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

from agent_framework import Agent, SkillsProvider, Message
from agent_framework.azure import AzureOpenAIChatClient

# 環境変数の読み込み
load_dotenv(override=True)

print("✅ インポート成功!")
print("📦 Agent Skills を利用するクラス:")
print("   - SkillsProvider: ファイルベースのスキルプロバイダー")
print("   - Agent: スキル付きエージェント")

In [ ]:
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}")
print(f"Deployment: {azure_deployment}")

## ステップ 2: OpenTelemetry トレーサーのセットアップ

Agent Skills の内部動作（`load_skill` / `read_skill_resource` ツール呼び出し）を
可視化するためにトレーシングを有効化します。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Metrics は無効化（必要に応じて変更）

configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## ステップ 3: スキルファイルの確認

まず、このワークショップに同梱されているスキルファイルの中身を確認しましょう。

In [ ]:
# スキルディレクトリの場所
skills_dir = Path("skills")

print(f"📂 スキルディレクトリ: {skills_dir.absolute()}")
print()

# ディレクトリ構造を表示
for skill_dir in sorted(skills_dir.iterdir()):
    if skill_dir.is_dir():
        print(f"📁 {skill_dir.name}/")
        for item in sorted(skill_dir.rglob("*")):
            if item.is_file():
                relative = item.relative_to(skill_dir)
                indent = "  " * (len(relative.parts) - 1)
                print(f"   {indent}📄 {relative}")

In [ ]:
# travel-planner の SKILL.md を読んでみる
skill_file = skills_dir / "travel-planner" / "SKILL.md"
content = skill_file.read_text(encoding="utf-8")

print("=" * 70)
print("📄 travel-planner/SKILL.md")
print("=" * 70)
print(content[:1500])  # 先頭部分を表示
if len(content) > 1500:
    print(f"\n... (残り {len(content) - 1500} 文字)")

## ステップ 4: SkillsProvider の作成

`SkillsProvider` は指定ディレクトリから `SKILL.md` を自動探索し、
スキルをエージェントに提供するプロバイダーです。

**内部動作:**
1. `skills/` ディレクトリを再帰探索（最大深度 2）
2. `SKILL.md` の YAML frontmatter（name, description）を解析
3. 参照されるリソースファイルの存在を検証
4. `load_skill` / `read_skill_resource` の 2 つのツールを自動生成
5. セッション開始時にスキル一覧をシステムプロンプトに注入

In [ ]:
# SkillsProvider を作成
skills_provider = SkillsProvider(skill_paths=str(skills_dir))

print("✅ SkillsProvider 作成完了")
print()

# 内部状態を確認
print(f"📦 発見されたスキル数: {len(skills_provider._skills)}")
for name, skill in skills_provider._skills.items():
    print(f"\n  🔧 {name}")
    print(f"     説明: {skill.description[:80]}...")
    print(f"     リソース: {[r.name for r in skill.resources]}")
    print(f"     ソース: {skill.path}")

print()
print(f"🔧 自動生成されたツール数: {len(skills_provider._tools)}")
for tool in skills_provider._tools:
    print(f"   - {tool.name}: {tool.description}")

In [ ]:
# システムプロンプトに注入される内容を確認
print("=" * 70)
print("📋 エージェントのシステムプロンプトに注入されるスキル広告:")
print("=" * 70)
print(skills_provider._instructions)

上記が **Advertise（広告）フェーズ** で注入される内容です。  
各スキルは `<skill>` タグで名前と説明だけ（~100トークン/スキル）が提示されます。  
エージェントはこの一覧を見て、必要に応じて `load_skill` → `read_skill_resource` を呼び出します。

## ステップ 5: スキル付きエージェントを実行（基本編）

スキルを持つエージェントに質問して、Progressive Disclosure の動作を確認しましょう。

エージェントが内部で行うこと:
1. 質問がスキルのドメインに合致 → `load_skill("travel-planner")` を自動呼び出し
2. 詳細な FAQ が必要 → `read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")` を呼び出し
3. 取得した知識に基づいて回答を生成

In [ ]:
# Azure OpenAI クライアントの作成
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

# スキル付きエージェントを作成
# context_providers にスキルプロバイダーを渡すだけ！
travel_agent = Agent(
    client=chat_client,
    name="TravelAssistant",
    instructions="あなたは親切な旅行アシスタントです。利用可能なスキルを活用してユーザーの質問に答えてください。日本語で回答してください。",
    context_providers=[skills_provider],
)

print("✅ スキル付きエージェント作成完了")
print(f"   名前: {travel_agent.name}")
print(f"   スキル: {list(skills_provider._skills.keys())}")

In [ ]:
# 例 1: 旅行スキルを呼び出す質問
print("=" * 70)
print("💬 例 1: 旅行プランニング")
print("=" * 70)

question = "2泊3日の京都旅行を計画しています。予算は5万円です。おすすめのプランを教えてください。"
print(f"\n🧑 ユーザー: {question}\n")

with tracer.start_as_current_span("SkillDemo-TravelPlan"):
    response = await travel_agent.run(question)
    print(f"🤖 エージェント:\n{response}")

In [ ]:
# 例 2: FAQ リソースを読み込む質問
print("=" * 70)
print("💬 例 2: 旅行 FAQ への質問")
print("=" * 70)

question2 = "海外旅行保険は必要ですか？また、スーツケースはどのサイズがいいですか？5泊の旅行です。"
print(f"\n🧑 ユーザー: {question2}\n")

with tracer.start_as_current_span("SkillDemo-TravelFAQ"):
    response2 = await travel_agent.run(question2)
    print(f"🤖 エージェント:\n{response2}")

### Jaeger でトレースを確認

Jaeger UI ([http://localhost:16686](http://localhost:16686)) でトレースを確認すると、
エージェントが以下のツールを順番に呼び出していることが分かります:

1. `load_skill("travel-planner")` — スキル本文を取得
2. `read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")` — FAQ を取得（必要な場合）
3. `read_skill_resource("travel-planner", "assets/itinerary-template.md")` — テンプレートを取得（必要な場合）

これが **Progressive Disclosure** の実際の動作です。

## ステップ 6: コードレビュースキルを試す

同じスキルプロバイダーには `code-reviewer` スキルも含まれています。  
エージェントは質問の内容に応じて、適切なスキルを自動選択します。

In [ ]:
# 例 3: コードレビュースキルを呼び出す質問
print("=" * 70)
print("💬 例 3: コードレビュー")
print("=" * 70)

code_question = """
以下の Python コードをレビューしてください:

```python
import sqlite3

def get_user(name):
    conn = sqlite3.connect('users.db')
    result = conn.execute(f"SELECT * FROM users WHERE name = '{name}'")
    data = result.fetchall()
    return data

def process_items(items):
    output = []
    for i in range(len(items)):
        output.append(items[i] * 2)
    return output
```
"""
print(f"🧑 ユーザー: {code_question}")

with tracer.start_as_current_span("SkillDemo-CodeReview"):
    response3 = await travel_agent.run(code_question)
    print(f"🤖 エージェント:\n{response3}")

## ステップ 7: マルチターン会話でスキルを活用

セッションを使ってマルチターン会話を行い、スキルの知識が保持されることを確認します。

In [ ]:
# マルチターン会話
print("=" * 70)
print("💬 マルチターン会話デモ")
print("=" * 70)

session = travel_agent.create_session()

conversation = [
    "3泊4日の台北旅行を計画しています。予算は8万円です。",
    "旅行日程のテンプレートを使って、具体的なプランを作ってください。",
    "持ち物で特に気をつけるべきことはありますか？",
]

with tracer.start_as_current_span("SkillDemo-MultiTurn"):
    for i, user_msg in enumerate(conversation, 1):
        print(f"\n{'─' * 70}")
        print(f"🧑 ターン {i}: {user_msg}")
        print(f"{'─' * 70}\n")
        
        response = await travel_agent.run(user_msg, session=session)
        print(f"🤖 エージェント:\n{response}\n")

## ステップ 8: スキルプロバイダーの内部動作を確認

`load_skill` と `read_skill_resource` の内部関数を直接呼び出して、戻り値を確認してみましょう。

In [ ]:
# load_skill の動作確認
print("=" * 70)
print("🔍 load_skill 関数の戻り値:")
print("=" * 70)
body = skills_provider._load_skill("travel-planner")
print(body[:800])
print(f"\n... (全体: {len(body)} 文字)")

print()

# read_skill_resource の動作確認
print("=" * 70)
print("🔍 read_skill_resource 関数の戻り値:")
print("=" * 70)
faq = await skills_provider._read_skill_resource("travel-planner", "references/TRAVEL_FAQ.md")
print(faq[:500])
print(f"\n... (全体: {len(faq)} 文字)")

print()

# 存在しないスキルを呼んだ場合
print("=" * 70)
print("🔍 存在しないスキルを呼び出した場合:")
print("=" * 70)
error_result = skills_provider._load_skill("nonexistent-skill")
print(f"結果: {error_result}")

## ステップ 9: カスタムスキルの作成方法

新しいスキルを作成する手順を確認します。以下の手順で自分のスキルを追加できます。

In [ ]:
import tempfile

# 新しいスキルをプログラム的に作成するデモ
print("=" * 70)
print("🛠️ カスタムスキルの作成デモ")
print("=" * 70)

# 一時ディレクトリにスキルを作成
with tempfile.TemporaryDirectory() as tmpdir:
    skill_dir = Path(tmpdir) / "greeting-helper"
    skill_dir.mkdir()
    
    # SKILL.md を作成
    skill_md = skill_dir / "SKILL.md"
    skill_md.write_text("""\
---
name: greeting-helper
description: 様々な言語での挨拶と文化的なマナーを教えます。海外旅行やビジネスでの挨拶に使用してください。
---

# 挨拶ヘルパー

## 基本的な挨拶

| 言語 | こんにちは | ありがとう | さようなら |
|------|-----------|-----------|----------|
| 英語 | Hello | Thank you | Goodbye |
| フランス語 | Bonjour | Merci | Au revoir |
| スペイン語 | Hola | Gracias | Adiós |
| 中国語 | 你好 | 谢谢 | 再见 |
| 韓国語 | 안녕하세요 | 감사합니다 | 안녕히 가세요 |

## マナー

- 日本: お辞儀の角度で敬意のレベルが変わる
- フランス: 頬へのキス (la bise) は親しい間柄で
- タイ: ワイ (合掌) で挨拶
""", encoding="utf-8")
    
    print(f"📄 スキルファイル作成: {skill_md}")
    print()
    
    # カスタムスキルでプロバイダーを作成
    custom_provider = SkillsProvider(skill_paths=tmpdir)
    
    print(f"✅ カスタムスキルプロバイダー作成完了")
    print(f"📦 発見されたスキル: {list(custom_provider._skills.keys())}")
    print()
    print("📋 注入されるプロンプト:")
    print(custom_provider._instructions)

## ステップ 10: `as_agent` メソッドでのスキル付与

`chat_client.as_agent()` でもスキルを付与できます。

In [ ]:
# as_agent メソッドでも context_providers を渡せる
agent_via_as_agent = chat_client.as_agent(
    name="SkillfulAgent",
    instructions="あなたは多機能アシスタントです。利用可能なスキルを活用して回答してください。日本語で回答してください。",
    context_providers=[skills_provider],
)

print("✅ as_agent メソッドでスキル付きエージェント作成完了")
print(f"   名前: {agent_via_as_agent.name}")
print(f"   スキル数: {len(skills_provider._skills)}")
print()

# テスト実行
with tracer.start_as_current_span("SkillDemo-AsAgent"):
    answer = await agent_via_as_agent.run(
        "Pythonのコードレビューで最も重要なセキュリティチェック項目を3つ教えてください。"
    )
    print(f"🧑 質問: Pythonのコードレビューで最も重要なセキュリティチェック項目を3つ教えてください。")
    print(f"\n🤖 回答:\n{answer}")

## まとめ

### 学んだこと

1. **Agent Skills の概念**
   - モジュラーな知識パッケージ（`SKILL.md` + リソースファイル）
   - [Agent Skills 仕様](https://agentskills.io/) に準拠
   - instructions への全知識記載からの脱却

2. **Progressive Disclosure パターン**
   - **Advertise**: 名前+説明をシステムプロンプトに注入（軽量）
   - **Load**: `load_skill` ツールで本文をオンデマンドロード
   - **Read Resources**: `read_skill_resource` ツールで FAQ/テンプレート取得

3. **SkillsProvider の使い方**
   ```python
   provider = SkillsProvider(skill_paths="skills/")
   agent = Agent(
       client=chat_client,
       instructions="...",
       context_providers=[provider],
   )
   ```

4. **スキルのディレクトリ構造**
   ```
   skills/
   └── my-skill/
       ├── SKILL.md           ← YAML frontmatter (name, description) + 本文
       ├── references/        ← 参照ドキュメント
       └── assets/            ← テンプレート等
   ```

5. **セキュリティ**
   - XML エスケープによるプロンプトインジェクション防御
   - パストラバーサル防御
   - シンボリックリンクエスケープ防御

### プロダクション Tips

- **スキル名**: 小文字英数字 + ハイフンのみ（例: `travel-planner`）
- **説明**: 1024 文字以内で、どんなタスクに使うかを明記
- **リソース**: FAQ やテンプレートは `references/` や `assets/` に整理
- **バージョン管理**: `metadata` の `version` フィールドでスキルのバージョンを管理
- **テスト**: 新しいスキルを作成したら、エージェントが正しくロードするか必ずテスト

---

### 次のステップ

- 自分のドメインに合わせたカスタムスキルを作成してみましょう
- マルチエージェントワークフロー（チュートリアル 05-07）とスキルを組み合わせてみましょう
- [Agent Skills 仕様](https://agentskills.io/) の詳細を確認しましょう

## 演習問題

1. **独自スキルの作成**: 自分の専門分野（料理レシピ、フィットネス、プログラミング言語など）の `SKILL.md` を作成し、エージェントに与えてみましょう

2. **複数リソースの活用**: references と assets の両方を持つスキルを作成し、エージェントが適切にリソースを使い分けるか確認しましょう

3. **スキルの自動選択**: 3つ以上のスキルを持つエージェントに、異なるドメインの質問を続けて投げ、正しいスキルが選ばれるか確認しましょう

In [ ]:
# 演習: 独自スキルを作成してテスト

async def exercise_custom_skill():
    """
    演習: 独自のスキルを作成してエージェントに与える
    
    手順:
    1. skills/ ディレクトリに新しいフォルダを作成（例: skills/cooking-helper/）
    2. SKILL.md を作成（YAML frontmatter + 本文）
    3. 必要に応じて references/ や assets/ にファイルを追加
    4. SkillsProvider で読み込み
    5. エージェントに質問してテスト
    """
    pass

print("🎯 演習準備完了 - 独自スキルを作成してみましょう!")